In [1]:
import dimod
from dwave.system import DWaveSampler
from dwave.system import EmbeddingComposite

In [2]:
# Aquí indicas los coeficientes del Hamiltoniano de Ising que resuelve el problema
# Tomamos H1 = Z0 Z1 + Z0 Z2
J = {(0,1):1, (0,2):1} # define los coeficientes Jjk 
h = {} # define los coeficientes hj
# los coeficientes no especificados toman el valor 0 por defecto
problem = dimod.BinaryQuadraticModel(h, J, 0.0, dimod.SPIN)
# el offset toma el valor 0.0: termino constante que se añade al Hamiltoniano
#dimod.SPIN: Parametro. -> los valores de las variables son 1, -1
print(f'The problem to solve is {problem}') 

The problem to solve is BinaryQuadraticModel({0: 0.0, 1: 0.0, 2: 0.0}, {(1, 0): 1.0, (2, 0): 1.0}, 0.0, 'SPIN')


EL RESULTADO ES: The problem to solve is BinaryQuadraticModel({0: 0.0, 1: 0.0, 2: 0.0}, {(1, 0): 1.0, (2, 0): 1.0}, 0.0, 'SPIN') que parece que se ha invertido lo que se ha introducido, pero en realidad Z0 Z1 = Z1 Z0 (es completamente simetrico), por lo que en realidad es lo mismo

AHORA SE CORRE EL PROCESO DE ANNEALING EN UNO DE LOS QUANTUM ANNEALERS

In [4]:
sampler = EmbeddingComposite(DWaveSampler())
result = sampler.sample(problem, num_reads=10)
print(f'The solution that we have obtained are {result}')

ValueError: API token not defined

Ha generado error porque se necesita un API

In [5]:
# METODO 2
# Solver exacto. Resuelve todas las combinaciones posibles -> preciso pero 
# solo para problemas pequeños porque crece exponencialmente

sampleset = dimod.ExactSolver().sample(problem)
print(sampleset)

   0  1  2 energy num_oc.
1 +1 -1 -1   -2.0       1
4 -1 +1 +1   -2.0       1
2 +1 +1 -1    0.0       1
3 -1 +1 -1    0.0       1
6 +1 -1 +1    0.0       1
7 -1 -1 +1    0.0       1
0 -1 -1 -1    2.0       1
5 +1 +1 +1    2.0       1
['SPIN', 8 rows, 8 samples, 3 variables]


In [6]:
# METODO 3
# Simulated Annealing: imita el comportamiento de un annealer cuantico
# la mejor opcion
# funciona localmente, escala mejor que exact solver, es lo mas parecido a usar Dwave sin hardware
from dwave.samplers import SimulatedAnnealingSampler

sampler = SimulatedAnnealingSampler()
result = sampler.sample(problem, num_reads=100)
print(result.first)

Sample(sample={0: np.int8(1), 1: np.int8(-1), 2: np.int8(-1)}, energy=np.float64(-2.0), num_occurrences=np.int64(1))


In [7]:
# METODO 4
# Tabu search: mas eficiente en algunos problemas grandes
from dwave.samplers import TabuSampler

sampler = TabuSampler()
result_tab = sampler.sample(problem, num_reads=100)

print(result_tab.first)

Sample(sample={0: np.int8(-1), 1: np.int8(1), 2: np.int8(1)}, energy=np.float64(-2.0), num_occurrences=np.int64(1), num_restarts=np.int64(0))


solo veo una solucion? para arreglarlo: 

In [8]:
sampler = SimulatedAnnealingSampler()
result = sampler.sample(problem, num_reads=100)

agg = result.aggregate() # aquí agrupo todos los resultados de las 100 lecturas
print(agg)

   0  1  2 energy num_oc.
0 -1 +1 +1   -2.0      58
1 +1 -1 -1   -2.0      42
['SPIN', 2 rows, 100 samples, 3 variables]


In [9]:
print(f'the best solution is: {result.first}')

the best solution is: Sample(sample={0: np.int8(-1), 1: np.int8(1), 2: np.int8(1)}, energy=np.float64(-2.0), num_occurrences=np.int64(1))
